# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR$^2$ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing entities by their `@id` as per the Croissant metadata specification.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display the dataset overview
meta = dataset.metadata
print('Dataset Name:', meta.name)
print('Description:', meta.description)
print('Version:', getattr(meta, 'version', 'N/A'))
print('License:', getattr(meta, 'license', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We will enumerate all record sets in the Croissant schema by their `@id`. For each record set, we'll list the fields and their types.

In [ ]:
# Display all record sets and their fields by @id
record_sets = dataset.record_sets()
print('Record sets found:')
for rs in record_sets:
    print(f"\nRecordSet: @id={rs['@id']}, name={rs['name']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            if isinstance(f, dict):
                print(f"  Field: @id={f.get('@id')}, name={f.get('name')}, dataType={f.get('dataType')}")
            else:
                print(f"  Field: @id={f}")
    else:
        print("  No fields enumerated.")

# Let's save the list of record set @ids for reference
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction

In this section, we demonstrate how to load records from a selected record set by `@id`.

Choose a record set (by `@id`) from above for demonstration and load it into a DataFrame. For this dataset, the main experimental record set typically contains protein abundance and attributes.

We will use the first record set found for illustration, but you can update the variable to use any `@id` as needed.

In [ ]:
# Choose a record set @id for further exploration (example uses the first found)
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"Selected RecordSet: {main_record_set_id}\n")

# Load records from all record sets into DataFrames by @id
dfs = dict()
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"@id: {rs_id}, loaded {len(df)} records.")
    except Exception as e:
        print(f"Failed to load records from @id={rs_id}: {e}")

if main_record_set_id:
    if main_record_set_id in dfs:
        print('\nColumns in the selected record set:')
        print(dfs[main_record_set_id].columns.tolist())
        display(dfs[main_record_set_id].head())
    else:
        print(f"No records found for record set {main_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate numeric field filtering, normalization, and grouping by a categorical attribute using only `@id` values for fields.

Update the numeric/categorical field `@id` as discovered from Section 2. We'll use the `@id` of the main numeric column (e.g., abundance, molecular weight, etc.) and a grouping field (e.g., description or protein class) by `@id`.

_Note: Replace `numeric_field_id` and `group_field_id` as appropriate for your dataset fields._

In [ ]:
# Example - set these with the field @id (from the record set field listing above)
# If unsure, run the previous cell to print columns and choose suitable fields.
numeric_field_id = None
group_field_id = None

df = dfs.get(main_record_set_id)

if df is not None:
    for col in df.columns:
        # Example heuristics: try detect reasonable defaults
        if numeric_field_id is None and (('abundance' in col.lower()) or ('molecular_weight' in col.lower()) or ('coverage' in col.lower()) or ('mw' in col.lower())):
            numeric_field_id = col
        if group_field_id is None and (('description' in col.lower()) or ('protein_class' in col.lower()) or ('group' in col.lower()) or ('category' in col.lower())):
            group_field_id = col

    print(f"Using numeric_field_id: {numeric_field_id}")
    print(f"Using group_field_id: {group_field_id}")

    # Convert numeric field to float
    if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean()  # for demonstration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > average:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print(f"Column {numeric_field_id} is not numeric or not found.")
else:
    print(f"No DataFrame found for record set {main_record_set_id}.")

## 5. Visualization

Visualize data distributions or relationships using standard Python plotting tools. Fields to plot must be referenced by their `@id`. If fields have not been set, assign them above accordingly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example histogram of the numeric field
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id exists, plot grouped boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to use `mlcroissant` to explore and process the [FAIR$^2$ mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) via its Croissant schema.
- Data entities were referenced using their `@id` as recommended.
- The notebook covered data loading, overview, extraction, basic EDA, and visualization.

You can further extend this notebook by exploring other record sets or fields, or by integrating with downstream ML workflows.